In [1]:
import pandas as pd
import sys
from pathlib import Path
import polars as pl
sys.path.append(str(Path("..").resolve()))
from src.services import bloomberg_client
from src.parsers import extract_ucf

In [10]:
nbf_file = Path(r"C:\Users\mmoin\PYTHON PROJECTS\Commissions Report\data\raw\Broker Data\NBF\NBF_Q1_2026.xlsx")
mydf = pd.read_excel(nbf_file, header=None)
mydf.columns = ["Region", "Branch"] + mydf.iloc[0, 2:].astype(str).str.replace(".","/").tolist()
df = pd.DataFrame(mydf.iloc[2:, :].values, columns=mydf.columns)
df = df[df.iloc[:, 0].astype(str).str.upper().isin(["GRAND TOTAL","TOTAL"]) == False]
df = df.loc[:, ~df.columns.str.upper().isin(["GRAND TOTAL","TOTAL"])]
df = df.reset_index(drop=True)

In [ ]:
df[df.columns[2:]] = df[df.columns[2:]].apply(pd.to_numeric, errors='coerce')
df = df.replace(r"^\s*$", pd.NA, regex=True)
df = df.dropna(how='all').reset_index(drop=True)
df

,Region,Branch,HHL,HPYM,HUTL,HPYT,HTA,HHL/U,HBF,HHIS,...,APLE,TRVL/U,HTA/B,HHIS/U,HBIX,SHPE,MSTY,NFLY,HPYT/B,RDDY
0,RN,Laval,13993322.52,26449199.60,0.00,81900.00,19812.00,1.402260e+05,780993.52,149996.00,...,0.00,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,B1,Vancouve,15750244.36,656201.23,2754105.75,2871553.23,2743598.78,1.007096e+06,1847283.87,1954584.24,...,0.00,0.00000,0.0,16660.374920,0.0,0.0,0.0,0.0,0.0,0.0
2,M2,PVM Mont,12483157.64,3233669.31,3772437.48,1524019.77,3547817.39,2.991566e+05,652255.39,152041.40,...,0.00,0.00000,9145.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,PR,Calgary,21057585.08,404596.99,1494010.80,49140.00,525463.77,7.243100e+05,537373.15,0.00,...,0.00,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,RS,DIX30,1032272.20,4025761.35,3261294.72,33742.80,538176.47,7.554248e+06,236865.50,231812.00,...,0.00,0.00000,11888.5,0.000000,0.0,6440.0,0.0,0.0,1774.8,0.0
5,PR,WINNIPEG,3344104.04,1084903.02,644327.28,2839857.93,487391.71,1.872713e+05,35208.40,111659.36,...,0.00,0.00000,0.0,0.000000,0.0,4427.5,0.0,0.0,0.0,0.0
6,B1,White Ro,8451251.38,0.00,439934.40,50778.00,125228.35,1.606009e+04,1015386.52,17532.00,...,0.00,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
7,TR,Toronto,3700279.48,68574.00,4029012.36,10114.65,335053.94,1.809437e+05,116247.06,0.00,...,0.00,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
8,PR,Medicine,216805.78,0.00,0.00,5588995.23,369130.58,0.000000e+00,0.00,0.00,...,0.00,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
9,RN,St-Jerom,2115644.98,3185677.90,0.00,0.00,135547.10,3.495746e+04,0.00,0.00,...,0.00,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df = pd.read_excel(r"C:\Users\mmoin\PYTHON PROJECTS\Commissions Report\data\raw\Broker Data\BMO\BMO_Q1_2026.xlsx", header=None)
search_grid = df.iloc[0:6,0:6].astype(str).apply(lambda x: x.str.strip().str.upper())
found = (search_grid == "BRANCH").stack()
if not found.any():
    raise ValueError("Could not find 'Branch' in the first 6 rows and columns.")
anchor_row, anchor_col = found.idxmax()
df_new = pd.DataFrame(data=df.iloc[anchor_row+1:, anchor_col:].values, columns=df.iloc[anchor_row, anchor_col:])
df_new = df_new.replace(r"^\s*$", pd.NA, regex=True)
df_new = df_new.dropna(how='all').reset_index(drop=True)
totals_mask = df_new["Branch"].astype(str).str.upper().isin(["TOTAL", "GRAND TOTAL", "TOTALS"])
if totals_mask.any():
    totals_idx = df_new[totals_mask].index[0]
    df_new = df_new.iloc[:totals_idx]
df_new = df_new.dropna(how='all').reset_index(drop=True)
# drop the column with "TOTAL" header if it exists
df_new = df_new.loc[:, ~df_new.columns.str.upper().isin(["TOTAL", "GRAND TOTAL", "TOTALS"])]

# df_new = df_new.replace(r"^\s*$", pd.NA, regex=True)
# df_new = df_new.dropna(how='all').reset_index(drop=True)
# df_new[df_new.columns] = df_new[df_new.columns].apply(pd.to_numeric, errors='coerce')
# df_new = df_new[df_new["Branch"].str.upper() != "TOTAL"].reset_index(drop=True)
# df_new.head()

3,Branch,Total,HHL,HHL.U,HBF,HTA,HUTL,HDIF,HHIS,HPYT,...,APLE,COSY,MSFH.U,RYHE,CRCY,TRVI,SOFY,CONY,AMZH.U,NaN
0,GTA WEST,56405.714,20742.765,11012.333,2599.807,5473.036,1539.525,737.66,1468.125,857.697,...,NaN,NaN,NaN,NaN,NaN,NaN,13.631,NaN,NaN,NaN
1,MONTREAL ST. JACQUES,31601.186,16514.97,868.434,2128.596,122.116,1925.892,2283.217,376.248,326.237,...,7.61,NaN,NaN,15.939,NaN,NaN,NaN,NaN,NaN,NaN
2,Laval,18759.998,12653.749,915.167,391.511,20.548,509.211,560.849,497.961,101.943,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.992,NaN,NaN
3,VAN SUBURBAN AND FRASER VALLEY,15752.102,3089.236,350.94,390.464,5170.552,864.398,20.755,621.254,23.607,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ottawa - West,15105.059,3972.027,9907.673,226.162,125.875,NaN,122.68,NaN,1.999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
